In [1]:
from glob import glob
import os
from os.path import join

import numpy as np
from o2_utils.selectors import find_files_from_pattern
from pathlib import Path

from multicamera_airflow_pipeline.jonah_241112.skeletons.defaults import kpt_dict, conf_thresholds_by_camera

In [2]:
def load_memmap_from_filename(filename):
    # Extract the metadata from the filename
    parts = filename.name.rsplit(".", 4)  # Split the filename into parts
    dtype_str = parts[-3]  # Get the dtype part of the filename
    shape_str = parts[-2]  # Get the shape part of the filename
    shape = tuple(map(int, shape_str.split("x")))  # Convert shape string to a tuple of integers
    # Load the array using numpy memmap
    array = np.memmap(filename, dtype=dtype_str, mode="r", shape=shape)
    return array

In [3]:
preds_2d_path = "/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions"
triang_path = "/n/groups/datta/kpts_pipeline/jonah_241112/results/triangulation"
# mice_to_use = ["J08101", "J08102"]  # need to compress these videos, nb might offset frames by 1 wrt already detected kpts :/ 
mice_to_use = ["J05102", "J05105"]
vid_dir_by_mouse = {
    "J08101": "/n/groups/datta/Jonah/20241029_PBN_Tac1_stim/raw_data/J08101",
    "J08102": "/n/groups/datta/Jonah/20241029_PBN_Tac1_stim/raw_data/J08102",
    "J05102": "/n/groups/datta/Jonah/20240520_vlPAG_Tac1_photom/20240520_6cam/data/J05102",
    "J05105": "/n/groups/datta/Jonah/20240520_vlPAG_Tac1_photom/20240520_6cam/data/J05105",
}
save_dir = "/n/groups/datta/Jonah/20240820_airflow_pipeline/more_training_data"

## Extract frames with low conf triangulations on paws

In [ ]:
# TODO (this ended up not being necessary)

# Extract frames with large jumps in distance

Video reading is sped up by having a few CPUs

In [4]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import videochef as vc
from tqdm.auto import tqdm
from PIL import Image

In [6]:
keypoint_names = list(kpt_dict.keys())
# paw_names = ["left_fore_paw", "right_fore_paw", "left_hind_paw_back", "right_hind_paw_back", "left_hind_paw_front", "right_hind_paw_front"]
paw_names = ["left_fore_paw", "right_fore_paw"]  # start with just forepaws, they're much harder.
jump_threshold = 30
cameras = ["bottom", "side1", "side2", "side3", "side4"]
n_frames_per_paw_per_camera = {"bottom": 150, "side": 30}

for mouse in mice_to_use:
# for mouse in ["J05105"]:
    sessions = glob(join(preds_2d_path, f"*{mouse}*"))
    
    for session in sessions:
        # if "20240702_J05105" not in session:
            # continue

        # Setup for this session
        base_name = os.path.basename(session)
        save_path = join(save_dir, "frames_with_large_jumps", base_name)
        os.makedirs(save_path, exist_ok=True)

        for camera in cameras:
            # Find file for this session / camera
            file = find_files_from_pattern(session, f"*.{camera}*.h5", error_behav="pass")
            if file is None: 
                continue
            print(file)
            raw_vid = glob(join(vid_dir_by_mouse[mouse], base_name, f"{base_name}*{camera}*.mp4"))[0]

            # Load data
            n_frames_per_paw = n_frames_per_paw_per_camera["side"] if "side" in camera else n_frames_per_paw_per_camera[camera]
            conf_thresholds = conf_thresholds_by_camera["side"] if "side" in camera else conf_thresholds_by_camera[camera]
            with h5py.File(file, "r") as h5f:
                keypoint_coords = np.array(h5f["keypoint_coords"])
                keypoint_conf = np.array(h5f["keypoint_conf"])
            
            # Find frames where keypoint speed is unusually high
            frames = []
            for paw in paw_names:
                coords = keypoint_coords[:, :, keypoint_names.index(paw)]
                confs = keypoint_conf[:, :, keypoint_names.index(paw)]
                this_conf_thresh = [val for kp_type,val in conf_thresholds.items() if kp_type in paw][0]
                coords[confs < (this_conf_thresh - 0.25)] = np.nan  # -0.25 b/c we want to capture mid-conf keypoints where the network was unsure.
                speeds = np.linalg.norm(np.diff(coords, axis=0, prepend=np.nan), axis=2)
                thresh = np.nanpercentile(speeds, 99.9)
                jump_frames = np.where(speeds > thresh)[0]

                # Often the frame with the large jump is actually the right one,
                # because it's jumping back to the correct keypoint.
                # So we allow a window of 5 frames around the jump frame, and then exclude
                # any frames that have quite high confidence so we're not sampling good frames.
                all_jump_frames = np.sort(np.concatenate([jump_frames + i for i in np.arange(-5, 5)]))
                corresponding_confs = confs[all_jump_frames].squeeze()
                all_jump_frames = all_jump_frames[corresponding_confs < (this_conf_thresh + 0.05)]
                frames.append(np.random.choice(all_jump_frames, n_frames_per_paw))
            frames = np.sort(np.concatenate(frames))  # frames must be sorted for videochef reader!!

            # Save frames
            with vc.io.VideoReader(raw_vid, frame_ixs=frames) as vr:
                for frame_ix, frame in tqdm(zip(frames, vr), total=len(frames)):
                    # save the frame from np array
                    Image.fromarray(frame).save(join(save_path, f"{base_name}.{camera}.frame_{frame_ix}.png"))


/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240626_J05102/20240626_J05102.bottom.0.h5


  0%|          | 0/300 [00:00<?, ?it/s]

/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240626_J05102/20240626_J05102.side1.0.h5


  0%|          | 0/60 [00:00<?, ?it/s]

/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240626_J05102/20240626_J05102.side2.0.h5


...

  0%|          | 0/60 [00:00<?, ?it/s]

/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240703_J05105/20240703_J05105.side3.0.h5


  0%|          | 0/60 [00:00<?, ?it/s]

/n/groups/datta/kpts_pipeline/jonah_241112/results/2D_predictions/20240703_J05105/20240703_J05105.side4.0.h5


  0%|          | 0/60 [00:00<?, ?it/s]